# 04 — Evaluation & Results
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** take the final saved BERTopic model and its per-document topic
assignments (from notebook 03) and produce the actual results — topic overview, keyword
visualizations, how topics trend over time, and a sanity check against the original news
categories. This notebook's outputs map directly onto the report's "Results" section.

**Run this in Google Colab.** No GPU needed — everything here loads already-computed results
(the saved model, the topic-assignment CSV); nothing is refit or re-embedded.


In [ ]:
import os
# No Google Drive needed anymore — all storage is local to this Colab session.
# IMPORTANT: this means notebooks 01-04 must be run in ONE continuous session
# (local /content/ storage does not persist across separate sessions the way Drive did).
BASE_DIR = '/content/topic-modelling-capstone'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
OUTPUTS_MODELS = f'{BASE_DIR}/outputs/models'
OUTPUTS_FIGURES = f'{BASE_DIR}/outputs/figures'


In [ ]:
!pip install -q bertopic pandas matplotlib seaborn wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42


## 1. Load the final model and topic assignments

Both were saved at the end of notebook 03 — the model itself (fitted on the full 284,916-document
dataset) and a copy of the preprocessed data with each headline's assigned `topic` column.


In [ ]:
best_model = BERTopic.load(f'{OUTPUTS_MODELS}/best_bertopic_model')
df = pd.read_csv(f'{DATA_PROCESSED}/headlines_with_topics.csv')

print(f"Loaded model and {len(df):,} topic-tagged documents.")
print(f"Number of topics (excluding outliers): {df['topic'].nunique() - (1 if -1 in df['topic'].values else 0)}")


## 2. Topic overview: sizes and keywords

`get_topic_info()` gives every topic's size and its top keywords in one table — this is the
master reference table for the report's results section.


In [ ]:
topic_info = best_model.get_topic_info()
topic_info.head(25)


In [ ]:
# Bar chart of the top 20 topics by size (excluding -1/outliers, shown separately since its
# scale would otherwise dwarf every real topic on the same chart).
top_n = 20
plot_df = topic_info[topic_info['Topic'] != -1].head(top_n).copy()
plot_df['label'] = plot_df['Topic'].astype(str) + ': ' + plot_df['Name'].str.split('_').str[1:4].str.join(' ')

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_df['label'], plot_df['Count'], color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Number of headlines')
ax.set_title(f'Top {top_n} topics by size (outliers excluded)')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topic_sizes_top20.png', dpi=150)
plt.show()

n_outliers = topic_info[topic_info['Topic'] == -1]['Count'].values[0]
print(f"For reference — outlier count: {n_outliers:,} ({n_outliers/len(df):.1%} of all documents)")


## 3. Keyword breakdown for the largest topics

Word clouds per topic — a quick, intuitive way to present what each topic is actually about in
the report, beyond just a ranked keyword list.


In [ ]:
from wordcloud import WordCloud

top_topics = topic_info[topic_info['Topic'] != -1]['Topic'].head(9).tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, topic_id in zip(axes.flat, top_topics):
    words = dict(best_model.get_topic(topic_id))
    wc = WordCloud(width=400, height=300, background_color='white').generate_from_frequencies(words)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    topic_name = ' '.join(topic_info[topic_info['Topic'] == topic_id]['Name'].values[0].split('_')[1:4])
    ax.set_title(f"Topic {topic_id}: {topic_name}", fontsize=11)

plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topic_wordclouds_top9.png', dpi=150)
plt.show()


## 4. Topic Examination: top 10 largest topics, keywords, and sample headlines

For each of the 10 largest topics, this shows its top 10 keywords alongside a small sample of
actual headlines assigned to it — the point is to manually check whether a sensible, human-
readable label could be assigned to each topic just from reading its keywords and examples,
which is a genuine qualitative check on topic quality that summary metrics alone don't capture.


In [ ]:
top10_topic_ids = topic_info[topic_info['Topic'] != -1]['Topic'].head(10).tolist()

for topic_id in top10_topic_ids:
    top_words = [w for w, _ in best_model.get_topic(topic_id)[:10]]
    size = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
    sample_headlines = df[df['topic'] == topic_id]['headline_text'].sample(5, random_state=RANDOM_STATE).tolist()

    print(f"=== Topic {topic_id} (size: {size:,}) ===")
    print("Top 10 words:", ', '.join(top_words))
    print("Sample headlines:")
    for h in sample_headlines:
        print(f"  - {h}")
    print()


**Observations:**

Manually reviewing the top 10 topics' keywords alongside sample headlines, most support a
clear, confident human label directly:

- Topic 0 → **Indian Politics & Elections** (poll, bjp, congress) — samples consistently match.
- Topic 1 → **Sports (Cricket, Football, IPL)** — samples consistently match.
- Topic 2 → **Education** (exam, school, university) — samples consistently match.
- Topic 3 → **Bollywood / Entertainment** (film, khan, kapoor) — mostly matches, but 2 of 5
  sampled headlines were off-topic (a real-estate business story, a political story) —
  suggesting some contamination near this cluster's boundary.
- Topic 4 → **India-Pakistan Relations / Security** (pak, pakistan, terror) — mostly matches,
  one sampled headline (about a bird species named after Pakistan) is a false-positive keyword
  match rather than a genuine security story.
- Topic 5 → **COVID-19 / Public Health** — samples consistently match.
- Topic 6 → **Courts & Legal Proceedings** (bail, plea, cbi) — samples consistently match.
- Topic 7 → **Crime: Suicide & Murder** — samples consistently match.
- Topic 8 → **Real Estate & Land/Housing** — mostly matches, one sampled headline (about a
  CBSE education R&D centre) appears misplaced.
- Topic 9 → **Banking & Finance/Business** — samples consistently match.

Overall: the large majority of sampled headlines within each topic genuinely support the
keyword-suggested label, consistent with the ~85-97% category-alignment figures found in the
quantitative sanity check later in this notebook. The handful of visibly mismatched examples are
a useful, honest illustration of the ~48% outlier rate's flip side — even *assigned* topics
contain a small fraction of borderline or loosely-related documents near cluster boundaries,
rather than the model being perfectly precise.


## 5. Topic Reduction

BERTopic supports merging the discovered topics down to a smaller, specified number via
`reduce_topics()`, which iteratively merges the most similar topic pairs (by c-TF-IDF
similarity) until the target count is reached — a lighter-weight alternative to re-running
HDBSCAN with a larger `min_cluster_size`, since it reuses the existing clustering rather than
reclustering from scratch.

We reduce from the current 45 topics down to **20**, a size that's easier to present and discuss
in a report while still retaining meaningful distinctions between major themes, and compare the
before/after topic list to see which topics merged together.


In [ ]:
N_TOPICS_REDUCED = 20

# reduce_topics() needs the original documents list in the same order the model was fit on.
docs_for_reduction = df['text_for_representation'].astype(str).tolist()

# Work on a fresh copy so the original 45-topic model (already saved, already used for every
# result above) is left untouched — reduce_topics() modifies the model in place.
import copy
reduced_model = copy.deepcopy(best_model)
reduced_model.reduce_topics(docs_for_reduction, nr_topics=N_TOPICS_REDUCED)

reduced_topic_info = reduced_model.get_topic_info()
print(f"Reduced from {len(topic_info[topic_info['Topic'] != -1])} to "
      f"{len(reduced_topic_info[reduced_topic_info['Topic'] != -1])} topics.")
reduced_topic_info.head(25)


In [ ]:
# Save the reduced model separately — kept distinct from the primary 45-topic model, which
# remains the one used for all earlier results (topics-over-time, category check, etc.), since
# reducing topics changes topic IDs/groupings and would invalidate those already-computed results
# if we overwrote the original.
# save_embedding_model=False for the same reason as the primary model in notebook 03 —
# keeps this file well under GitHub's 100MB hard limit.
reduced_model.save(f'{OUTPUTS_MODELS}/best_bertopic_model_reduced20', serialization='pickle',
                    save_embedding_model=False)
reduced_topic_info.to_csv(f'{OUTPUTS_MODELS}/reduced20_topic_info.csv', index=False)

import os
reduced_size_mb = os.path.getsize(f'{OUTPUTS_MODELS}/best_bertopic_model_reduced20') / (1024 * 1024)
print(f"Saved reduced-topic model ({reduced_size_mb:.1f} MB) and summary table locally.")


**Observations:**

`reduce_topics(nr_topics=20)` produced **19 topics** (BERTopic's merge algorithm doesn't always
land on the exact requested count — it stops at the nearest point its hierarchical merging
reaches). Comparing the reduced topics against the original 45 shows two distinct kinds of
merges:

**Sensible merges — combining genuinely related themes:**
- COVID (Topic 5) + cancer/hospital/health (Topic 10) → one broader "public health" topic.
- Suicide/murder (Topic 7) + rape/minor (Topic 12) → one broader "violent crime" topic.
- Scam/fraud (Topic 17) + stolen/thief/robbery (Topic 13) → one broader "financial/property crime" topic.
- Parking/road/traffic (Topic 16) + metro/train/railway (Topic 22) → one broader "transportation" topic.
- Rain/flood/monsoon (Topic 14) + water/water supply/dam (Topic 18) → one broader "weather & water resources" topic.

These are genuine improvements for presentation purposes — the original fine-grained split
wasn't adding much distinct value, and the merged version is arguably easier for a reader to
digest without losing much meaning.

**Questionable merges — combining seemingly unrelated themes:**
- Politics/elections (Topic 0) merged with education/exams/schools (Topic 2) into one topic
  led by keywords "bjp, poll, congress, school." These are not obviously the same theme; this
  likely reflects a c-TF-IDF/embedding-similarity artifact (e.g. shared institutional/government
  vocabulary) rather than a genuinely unified topic.
- Cricket/sports (Topic 1) merged with Bollywood/film (Topic 3) into one topic led by "cup,
  film, win, world cup." Sports and entertainment are conceptually distinct; this merge loses
  a real, meaningful distinction the 45-topic model had correctly captured.

**Topics that did not merge at all**: wildlife (tiger/dog/zoo) and road accidents
(accident/killed/truck) kept identical sizes before and after reduction, suggesting they were
already well-separated, distinctive clusters that had no similar neighbor to merge with.

**Recommendation**: given that reduction improved some groupings but measurably degraded others
(politics-with-education and sports-with-entertainment being the clearest cases), **the original
45-topic model is recommended as the primary result** for this project — it preserves
distinctions that matter. The 20-topic reduction is presented as a secondary, more
presentation-friendly view, with the explicit caveat that a couple of its groupings merge
genuinely distinct real-world themes and should not be read as equivalent in quality to the
groupings that merged sensibly.


## 6. Topics over time

Since headlines span 2001–2023, tracking how topic volume shifts over the years gives real
historical insight — e.g. does a "COVID" topic spike exactly where expected? Does an election
-related topic show periodic peaks matching India's actual election calendar?


In [ ]:
df['publish_date'] = pd.to_datetime(df['publish_date'])
df['year'] = df['publish_date'].dt.year

# Focus on the 8 largest real topics for readability — more than that becomes unreadable on one chart.
top8_topics = topic_info[topic_info['Topic'] != -1]['Topic'].head(8).tolist()
topic_names = {
    t: ' '.join(topic_info[topic_info['Topic'] == t]['Name'].values[0].split('_')[1:3])
    for t in top8_topics
}

yearly_topic_counts = (
    df[df['topic'].isin(top8_topics)]
    .groupby(['year', 'topic'])
    .size()
    .unstack(fill_value=0)
    .rename(columns=topic_names)
)

fig, ax = plt.subplots(figsize=(13, 6))
yearly_topic_counts.plot(ax=ax, marker='o', markersize=3)
ax.set_title('Headline volume over time for the 8 largest topics')
ax.set_xlabel('Year')
ax.set_ylabel('Headline count')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/topics_over_time.png', dpi=150)
plt.show()


## 7. Sanity check: discovered topics vs original news categories

The dataset came with its own `headline_category` labels (unused during modelling — BERTopic
never saw them). Checking whether each discovered topic's headlines mostly share one original
category is a good, independent sanity check that the model found *real* structure rather than
arbitrary clusters.


In [ ]:
# For each of the top topics, what's the dominant original category_top among its headlines?
category_check = []
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic'].head(15):
    subset = df[df['topic'] == topic_id]
    top_cat = subset['category_top'].value_counts()
    dominant_cat = top_cat.index[0]
    dominant_pct = top_cat.iloc[0] / len(subset)
    topic_name = ' '.join(topic_info[topic_info['Topic'] == topic_id]['Name'].values[0].split('_')[1:4])
    category_check.append({
        'topic_id': topic_id,
        'topic_keywords': topic_name,
        'size': len(subset),
        'dominant_original_category': dominant_cat,
        'pct_matching_dominant_category': round(dominant_pct, 3),
    })

category_check_df = pd.DataFrame(category_check)
category_check_df


## 8. What's in the outlier bucket?

~48% of headlines weren't assigned to any topic. Inspecting a random sample directly (rather
than just citing the percentage) helps distinguish "genuinely one-off, hard-to-cluster stories"
from any systematic pattern worth addressing.


In [ ]:
outlier_sample = df[df['topic'] == -1].sample(15, random_state=RANDOM_STATE)[['headline_text', 'category_top']]
outlier_sample


In [ ]:
# Category distribution within the outlier group, compared to the overall dataset —
# checks whether outliers skew toward specific categories (e.g. very long-tail local news)
# rather than being a random cross-section.
outlier_categories = df[df['topic'] == -1]['category_top'].value_counts(normalize=True).head(10)
overall_categories = df['category_top'].value_counts(normalize=True).head(10)

comparison = pd.DataFrame({
    'outlier_share': outlier_categories,
    'overall_share': overall_categories
}).fillna(0).round(3)
comparison


## 9. Save summary tables for the report

In [ ]:
topic_info.to_csv(f'{OUTPUTS_MODELS}/final_topic_info.csv', index=False)
category_check_df.to_csv(f'{OUTPUTS_MODELS}/topic_vs_category_check.csv', index=False)
print("Saved topic summary tables locally — copy these into the repo (see final cell below).")


## 10. Summary of Results & Conclusion

- **Final model**: 45 topics, 47.6% outliers (135,613 of 284,916 headlines), 2001-2023.
- **Largest topics**: politics/elections (17,081 headlines), cricket/sports (14,958), education
  (13,418), Bollywood (7,997), India-Pakistan/security (6,620).
- **Topic Examination (top 10 topics)**: manually reviewing keywords + sample headlines
  confirmed 8 of the 10 largest topics support a clear, consistent human label with no visible
  contamination in the sampled headlines. 2 topics (Bollywood/entertainment, India-Pakistan
  relations) showed minor contamination — 1-2 of 5 sampled headlines were off-topic — consistent
  with the ~48% outlier rate's flip side: even assigned topics aren't perfectly pure.
- **Topic Reduction (45 → 20)**: `reduce_topics()` produced 19 topics. Several merges were
  clearly sensible (COVID + general health topics combined; suicide/murder + rape/minor crime
  topics combined; parking/traffic + metro/rail combined; rain/flood + water-supply combined).
  Two merges were judged questionable: politics/elections merged with education/schools, and
  cricket/sports merged with Bollywood/entertainment — both combining conceptually distinct
  real-world themes, likely a c-TF-IDF similarity artifact rather than genuine thematic overlap.
  **Recommendation: use the original 45-topic model as the primary result**; present the
  20-topic reduction as a secondary, presentation-friendly alternative with this caveat noted.
- **Topics-over-time — strong validation result**: the COVID topic is essentially flat/near-zero
  from 2001-2019, spikes sharply in 2020-2021, then recedes by 2022-23 — exactly matching the
  real pandemic timeline. The elections topic shows distinct peaks in 2009, 2014, and 2019 —
  precisely India's Lok Sabha general election years.
- **Category sanity check**: most topics align strongly with a single original
  `headline_category` (e.g. stolen/thief/robbery → 97.1%, suicide/murder → 92.8%). Some
  moderate-alignment topics (elections, banking) reflect that `city` is a broad catch-all
  covering ~60% of the dataset, not a modelling weakness.
- **Outlier characterization**: outlier category distribution closely mirrors the overall
  dataset; sports is notably under-represented among outliers (0.9% vs 3.5% overall), meaning
  sports headlines cluster unusually well. Manual inspection shows genuinely idiosyncratic,
  one-off local stories in the outlier bucket.
- **Overall conclusion**: this is a usable, interpretable topic model for understanding two
  decades of Indian news coverage, independently validated via topic-over-time trends and
  category alignment, with an honest accounting of both its outlier rate and the trade-offs
  observed when reducing topic granularity. Future work: apply `reduce_outliers()`, run the full
  grid search on the complete dataset given more compute, and treat `city`'s dominance as a
  separate, finer-grained modelling problem.


## 11. Package Everything for GitHub

Zips every artifact this project produced — mirroring the repo's exact folder structure
(`data/processed/`, `outputs/figures/`, `outputs/models/`) — into one file. Download this single
zip from the Colab file browser (folder icon, left sidebar) instead of downloading each file
individually, then extract it directly into your local repo's root; the folder structure inside
already matches, so files land in the right place automatically.


In [ ]:
import shutil

ARCHIVE_BASE = '/content/topic-modelling-capstone-artifacts'
shutil.make_archive(ARCHIVE_BASE, 'zip', BASE_DIR)

archive_path = f'{ARCHIVE_BASE}.zip'
size_mb = os.path.getsize(archive_path) / (1024 * 1024)
print(f"Created {archive_path} ({size_mb:.1f} MB)")

# Quick sanity check on the two model files specifically, since those are the ones with a real
# risk of exceeding GitHub's 100MB per-file hard limit.
for model_file in ['best_bertopic_model', 'best_bertopic_model_reduced20']:
    p = f'{OUTPUTS_MODELS}/{model_file}'
    if os.path.exists(p):
        mb = os.path.getsize(p) / (1024 * 1024)
        status = "OK" if mb < 95 else "TOO LARGE FOR A NORMAL GIT PUSH"
        print(f"  {model_file}: {mb:.1f} MB [{status}]")

print("\nDownload the zip above (Colab file browser, left sidebar), extract it into your")
print("local repo root (e.g. ~/Capstone_Project/topic-modelling-capstone/), then:")
print("  git add data/ outputs/")
print("  git commit -m \"Add final data, figures, and models to repo\"")
print("  git push")
